# Idealized dune transport tutorial

This tutorial illustrates the behaviour of the exnerFoam solver through the migration of an idealized two-dimensional dune. The objective is to demonstrate the coupling between sediment transport and bed evolution while keeping the hydrodynamic description deliberately simple.

The solver computes the evolution of the bed elevation $z_b$ by solving the Exner equation, which expresses the conservation of bed material:
$$\frac{\partial z_b}{\partial t} + \nabla.q_b=0$$
where $q_b$ is the volumetric bedload transport rate per unit width.
In this simplified example, the hydrodynamics are not solved explicitly. Instead, the flow discharge per unit width $Q$ is assumed constant throughout the domain. Neglecting vertical velocity variations, the depth-averaged velocity is estimated from the local flow depth as:
$$U = \frac{Q}{H-z_b}$$
where HHH is the water surface elevation, assumed constant, and $H-z_b$ therefore represents the local water depth.

The bedload transport rate is prescribed through a power-law relationship:
$$q_b=\alpha U^\beta$$
where $\alpha$ and $\beta$ are empirical coefficients controlling the magnitude and nonlinearity of the transport response.

Combining these expressions yields
$$\frac{\partial z_b}{\partial t} + c(z_b)\nabla.z_b=0$$
with
$$c(z_b)=\alpha\beta\frac{Q^\beta}{\left(H-z_b\right)^{\beta+1}}$$
The resulting equation is a nonlinear advection equation in which the propagation speed depends on the local bed elevation. Regions of higher bed elevation migrate faster because the local flow depth is reduced, increasing both the velocity and the sediment transport rate. This mechanism leads naturally to the downstream migration of dunes while preserving their overall shape.

This highly simplified formulation is not intended to reproduce realistic river hydraulics. Its purpose is solely to illustrate the numerical behaviour of the Exner equation decoupled with the hydrodynamics.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from contextlib import redirect_stdout
import os

from simusIdealizedDune import DuneCase1D, zb0_gaussian, bedformCelerity

plt.rcParams["font.size"] = 15
plt.rcParams["lines.linewidth"] = 2.5

pathCase0 = "./case00"

## Running a first simulation

We start by defining the computational domain, the initial dune geometry, the sediment transport parameters, and the numerical settings of the simulation.

The computational domain is a one-dimensional channel of length $L_x$ and constant water surface elevation $H$ (Hz in the code that follows). An initially Gaussian-shaped dune is placed near the upstream part of the domain. The dune evolution is computed on a finite-area mesh composed of $n_x$ cells.

The bedload transport law is controlled by the coefficients $\alpha$ and $\beta$, while the temporal evolution is advanced using an Adams-Bashforth scheme (order 1 or 2). Different discretization schemes can be selected for the bedload flux divergence, allowing the user to investigate the trade-off between numerical diffusion and solution accuracy.

In [ ]:
# domain dimension
Lx = 6.  # domain length
Hz = 1.  # domain height

# dune initial condition
duneHeight = 0.1  # initial dune height
duneCrestPos = 2.  # initial position of dune crest
duneWidth = 0.6  # parameter linked to dune width

# water discharge per unit width
Qwater = 1.

#spatial discretization, number of faces making up the finite-area mesh
nx = 600
# time step
dt = 0.05

# coefficients for bedload flux
alpha, beta = 0.05, 1.5

# scheme for bedload flux divergence
# options are linear, upwind and linearUpwind
qbScheme = "upwind"
# order of Adams-Bashforth temporal scheme, 1 or 2
ABorder = 2

bedloadDict = {
    "alpha": alpha,
    "beta": beta,
    "qbScheme": "linearUpwind",
    "ABorder": 1
}

## Creating and running the OpenFOAM case

The simulation is performed using the DuneCase1D helper class, which automatically generates all OpenFOAM input files required by the exnerFoam solver.

The workflow consists of four steps:

- Define the case directory.
- Create and configure the OpenFOAM case.
- Initialize the bed elevation with the prescribed dune shape.
- Run the simulation.

In [ ]:
# choose a location for the OpenFOAM idealized dune transport case
nameCase = f"duneCase"
pathCase = f"./{nameCase}"

# create and configure the case
duneCase = DuneCase1D(pathCase0, pathCase, dt, nx, Lx, Hz, Qwater, bedloadDict)

duneCase.setup()

# set the initial dune geometry
duneCase.setInitialCondition(duneHeight, duneCrestPos, duneWidth)

# run the simulation
duneCase.runSim()

## Comparing the numerical and analytical solutions

Because the governing equation can be written as a nonlinear advection equation,
$$\frac{\partial z_b}{\partial t} + c(z_b)\frac{\partial z_b}{\partial x}=0\,,$$

the solution can be interpreted using the method of characteristics. Each bed elevation value propagates at a local celerity
$$c(z_b)=\alpha\beta\frac{Q^\beta}{(H−z_b)^{\beta+1}}\,.$$

In the absence of shock formation, the analytical solution is obtained by transporting the initial bed profile along these characteristic curves. The numerical results produced by exnerFoam can therefore be directly compared with this reference solution.

The following figure shows the bed elevation at the different saved time steps. Solid coloured lines correspond to the numerical solution, while the black dotted curves represent the analytical solution obtained from the characteristic advection.

In [ ]:
fig, axZb = plt.subplots(figsize=(12, 5))

# list of time available
timeList = ["0"] + duneCase.getTimes()
ntimes = len(timeList)

colors = ["cornflowerblue", "tomato"]
cm = LinearSegmentedColormap.from_list(
    "Custom", colors, N=ntimes)
colors = cm(np.arange(0, cm.N))

nxa = 500  # number of points to represent analytical solution
Xa0 = np.linspace(0, Lx, nxa)  # initial grid for solution
Zba = zb0_gaussian(Xa0, duneHeight, duneCrestPos, duneWidth)
Ca = bedformCelerity(duneCase, Zba)

for i, time in enumerate(timeList):
    Xbed, Ybed, Zbed = duneCase.getFaceCenters(time=time)
    axZb.plot(Xbed, Zbed, color=colors[i], lw=2.5, label=r"$z_b(t=$" + time + r" $s)$")
    # plot analytical solution
    # x-positions after advection along characteristic lines
    XaT = Xa0 + Ca * float(time)
    axZb.plot(XaT, Zba, ls="dotted", lw=3, color="black")

legend_elements = [
    Line2D([0], [0], ls="dotted", lw=2.5, color="black", label="solution")
]

handles, labels = axZb.get_legend_handles_labels()

axZb.legend(handles=handles+legend_elements, loc="upper left")
axZb.set_xlim(0.5, 5)
axZb.set_ylabel(r"$z_b\,[m]$")
axZb.set_xlabel(r"$x\,[m]$")
axZb.grid()

fig.tight_layout()

plt.show()

duneCase.delete()

The dune migrates downstream while maintaining approximately its initial shape. The crest moves faster than the troughs because the local celerity $c(z_b)$ increases with bed elevation. Consequently, the higher portions of the dune experience larger transport rates and propagate more rapidly.

The close agreement between the numerical and analytical solutions demonstrates that the solver correctly reproduces the characteristic propagation of the Exner equation. Small discrepancies may arise from numerical diffusion introduced by the spatial discretization scheme, particularly when using first-order upwind interpolation.

The solution is not translated at a uniform velocity. Each elevation is advected at a different speed, which is the hallmark of a nonlinear conservation law. This behaviour ultimately leads to steepening of the dune front and may produce shock-like features for sufficiently large amplitudes or long simulation times.

## Influence of the numerical schemes

The accuracy of the Exner equation solution depends not only on the spatial and temporal resolution but also on the numerical schemes used to discretize the governing equations. In particular, the choice of the bedload flux interpolation scheme and the temporal integration method can significantly affect numerical diffusion, phase errors, and overall solution accuracy.

In this section, several combinations of spatial and temporal discretization schemes are compared. The objective is to assess their ability to reproduce the analytical solution presented previously and to highlight the trade-off between numerical stability and accuracy.

The following bedload flux interpolation schemes are considered:
- linear: second-order central interpolation, generally less diffusive but potentially more oscillatory;
- upwind: first-order upwind interpolation, robust but significantly more diffusive;
- linearUpwind: second-order upwind-biased interpolation, providing a compromise between stability and accuracy.

Two temporal integration schemes are also investigated, both being explicit schemes:
- Euler explicit (first-order accurate);
- Adams-Bashforth 2 (second-order accurate).

The physical parameters are kept identical for all simulations so that the observed differences can be attributed solely to the numerical discretization.

In [ ]:
# sensitivity to numerical schemes, scheme options
ABorders = {1: "Euler explicit", 2: "Adams-Bashforth 2"}
qbSchemes = ("linear", "upwind", "linearUpwind")

# rest of parameters are identical for every case
# domain dimension
Lx = 6.  # domain length
Hz = 1.  # domain height

# dune initial condition
duneHeight = 0.1  # initial dune height
duneCrestPos = 2.  # initial position of dune crest
duneWidth = 0.6  # parameter linked to dune width

# water discharge per unit width
Qwater = 1.

# spatial discretization, number of faces making up the finite-area mesh
nx = 200
# time step
dt = 0.1

# coefficients for bedload flux
alpha, beta = 0.05, 1.5

In [ ]:
# results are plotted at each time written in the list
timeResList = ["16", "24"]  # latestTime not supported
ntimesRes = len(timeResList)

colorList = ["#009E71", "#0072B2", "#D55E00", "#56B4E9"]
lsList = ["solid", "dashed", "dashdot", "dotted"]

# Analytical solution
nxa = 500  # number of points to represent analytical solution
Xa0 = np.linspace(0, Lx, nxa)  # initial grid for solution
Zba = zb0_gaussian(Xa0, duneHeight, duneCrestPos, duneWidth)

duneCaseList = []

fig, axs = plt.subplots(2, ncols=ntimesRes, figsize=(7 * ntimesRes, 10))

for i, ABorder in enumerate(ABorders.keys()):
    for k, qbScheme in enumerate(qbSchemes):
        bedloadDict = {
            "alpha": alpha,
            "beta": beta,
            "qbScheme": qbScheme,
            "ABorder": ABorder
        }
        nameCase = f"duneCase_AB{ABorder}_qb{qbScheme}"
        pathCase = f"./{nameCase}"
        label = qbScheme
        if label == "linearUpwind":
            label = "linear upwind"
        # create the case
        duneCase = DuneCase1D(pathCase0, pathCase, dt, nx, Lx, Hz, Qwater, bedloadDict)
        duneCase.setup()
        duneCase.setInitialCondition(duneHeight, duneCrestPos, duneWidth)
        # run the simulation
        duneCase.runSim()
        for j, timeRes in enumerate(timeResList):
            Xbed, Ybed, Zbed = duneCase.getFaceCenters(time=timeRes)
            axs[i, j].plot(
                Xbed, Zbed, color=colorList[k], lw=2.5, ls=lsList[k], label=label)
            # plot analytical solution
            if k == len(qbSchemes) - 1:
                Ca = bedformCelerity(duneCase, Zba)
                XaT = Xa0 + Ca * float(timeRes)
                axs[i, j].plot(
                    XaT, Zba, color="black", ls="dotted",
                    lw=3, label="analytical solution"
                )
                title = ABorders[ABorder] + f", time = {timeRes} s"
                axs[i, j].set_title(title)
        duneCaseList.append(duneCase)

for duneCase in duneCaseList:
    duneCase.delete()

axs[0, 0].legend(fontsize=12, loc="upper left")

for ax in fig.axes:
    ax.set_xlim(2, 5)
    ax.set_ylim(0, 0.14)
    ax.set_xlabel(r"$x\,[m]$")
    ax.set_ylabel(r"$z_b\,[m]$")
    ax.grid()

fig.tight_layout()

plt.show()

## Discussion

Several conclusions can typically be drawn from this comparison:

The upwind scheme is the most diffusive. The dune crest tends to flatten and the profile becomes progressively smoother over time.
The linear scheme generally provides the sharpest representation of the dune shape, yielding the closest agreement with the analytical solution.
The linearUpwind scheme usually represents a compromise between robustness and accuracy, reducing numerical diffusion while maintaining stable solutions.
Differences between Euler explicit and Adams-Bashforth 2 are generally less pronounced than those associated with the spatial discretization. However, the second-order temporal scheme can improve phase accuracy and reduce temporal truncation errors.

In this benchmark, the spatial discretization of the bedload flux has a greater impact on the solution quality than the choice of temporal integration scheme. Numerical diffusion introduced by the flux interpolation scheme is the dominant source of error and directly affects the preservation of the dune shape.

## Mesh convergence analysis

The previous section highlighted qualitative differences between the available numerical schemes. To assess these differences more rigorously, a mesh convergence study is performed.

For each combination of temporal and spatial discretization schemes, simulations are repeated on a sequence of increasingly refined meshes. The numerical solution at a fixed time is compared against the analytical solution, and the error is quantified using the Root Mean Square Error (RMSE):
$$RMSE=\sqrt{\frac{1}{N_f}\sum_{f=0}^{N_f} \left( z_{b,f}^{num} - z_{b,f}^{ana} \right)}$$
where $z_{b,f}^{num}$ and $z_{b,f}^{ana}$ are respectively the numerical and analytical solution at face f.

The RMSE provides a global measure of the discrepancy between the numerical and analytical bed profiles. A convergent numerical scheme should exhibit decreasing errors as the mesh is refined.

To facilitate interpretation, the upper horizontal axis is expressed in terms of the Courant number based on the maximum dune celerity,
$$Co=\frac{c(hd)\delta t}{\delta x}\,,$$

where $h_d$ is the initial dune height and $c(h_d)$ is the corresponding propagation velocity. Since the time step is kept constant, increasing the mesh resolution also increases the Courant number.

In [ ]:
# sensitivity to numerical schemes, scheme options
ABorders = {1: "Euler explicit", 2: "Adams-Bashforth 2"}
qbSchemes = {"linear": ("#D55E00", "s"), "upwind": ("#0072B2", "o"), "linearUpwind": ("#009E71", "d")}

# sensitibity to grid size
nxList = [10, 20, 30, 50, 100, 200, 300, 500, 700, 1000]

dt = 0.05  # time step value
timeRes = "20"


def RMSE(X, Y):
    """compute RMSE"""
    n = len(X)
    rmse = np.sqrt((1/n) * np.sum((X-Y)**2))
    return rmse


# analytical solution on fine grid
Xa0 = np.linspace(-2, 8, 2000)
Zba = zb0_gaussian(Xa0, duneHeight, duneCrestPos, duneWidth)

fig, axs = plt.subplots(ncols=2, figsize=(8.3, 5))

for i, ABorder in enumerate(ABorders.keys()):
    for j, qbScheme in enumerate(qbSchemes.keys()):
        bedloadDict = {
            "alpha": alpha,
            "beta": beta,
            "qbScheme": qbScheme,
            "ABorder": ABorder
        }
        rmseValues = []
        label = qbScheme
        if label == "linearUpwind":
            label = "linear upwind"
        for nx in nxList:
            nameCase = f"duneCase_AB{ABorder}_qb{qbScheme}_nx{nx}"
            pathCase = f"./{nameCase}"
            print(f"Create and run case: {pathCase}")
            duneCase = DuneCase1D(pathCase0, pathCase, dt, nx, Lx, Hz, Qwater, bedloadDict)
            # avoid standard output of the following methods
            with open(os.devnull, "w") as f:
                with redirect_stdout(f):
                    duneCase.setup()
                    duneCase.setInitialCondition(duneHeight, duneCrestPos, duneWidth)
                    duneCase.runSim()
                    Xbed, Ybed, Zbed = duneCase.getFaceCenters(time=timeRes)
            Ca = bedformCelerity(duneCase, Zba)
            XaT = Xa0 + Ca * float(timeRes)
            Cbed = bedformCelerity(duneCase, Zba)
            XTsol = Xa0 + Cbed * float(timeRes)
            # interpolate analytical solution on same grid
            zbsol_interp = np.interp(Xbed, XTsol, Zba)
            rmseValues.append(RMSE(Zbed, zbsol_interp))
            cMaxDune = bedformCelerity(duneCase, duneHeight)
            duneCase.delete()
        color, marker = qbSchemes[qbScheme]
        axs[i].scatter(nxList, rmseValues, color=color, marker=marker, label=label)

axs[0].text(
    200, 0.012, "Euler explicit", fontsize=16,
    bbox=dict(
        boxstyle="round",
        ec="gray",
        fc="white",
    )
)

axs[1].text(
    60, 0.012, "Adams-Bashforth 2", fontsize=16,
    bbox=dict(
        boxstyle="round",
        ec="gray",
        fc="white",
    )
)


def nxToCO(nx):
    return nx * dt * cMaxDune / Lx


def COToNx(co):
    return (Lx * co) / (dt * cMaxDune)

for ax in axs:
    ax.set_xlabel(r"$n_x$")
    ax.tick_params(
        axis="both", which="major")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.grid()
    axTop = ax.secondary_xaxis("top", functions=(nxToCO, COToNx))
    axTop.set_xlabel(r"$c(z_b=h_d) dt/dx$")

axs[0].set_ylabel("RMSE")
axs[0].legend(loc="lower left", fontsize=16)
axs[0].set_ylim(axs[1].get_ylim())
axs[1].tick_params(left=False, labelleft=False)

fig.tight_layout()

plt.show()

## Discussion

Several trends can generally be observed from the convergence curves.

- The RMSE decreases as the number of cells increases, confirming that the numerical solution converges toward the analytical solution.
- For high Courant number ($Co > 1$), oscillation appears in the simulation and the numerical solution blows up leading sharp increase in RMSE.
- The upwind interpolation scheme exhibits the largest errors on coarse meshes due to its inherent numerical diffusion.
- The linear and linearUpwind schemes achieve significantly lower errors for a given mesh resolution, demonstrating the benefit of higher-order spatial discretization.
- For sufficiently fine meshes, the differences between spatial schemes become smaller and the error is increasingly controlled by temporal discretization and floating-point precision.
- The use of the Adams-Bashforth 2 scheme generally improves accuracy compared with the first-order explicit Euler method, although the impact remains smaller than that of the flux interpolation scheme.

The results indicate that the spatial discretization of the bedload flux is the dominant source of numerical error for this benchmark problem. In particular, the first-order upwind scheme introduces substantial artificial diffusion that smooths the dune profile and slows convergence.